# Phase-4: Risk Index Construction (London-final-light)

**Goal:** Build a single, empirically justified risk score per London LSOA.

**London-final-light excludes `stop_search_rate` and `seasonal_volatility`.** The feature set is read directly from the Phase 3 validated parquet, so this notebook adapts to whatever Phase 3 retained.

- **4a. Weight derivation:** Negative Binomial Regression (primary) + Random Forest (cross-validation)
- **4b. Build the index:** Normalise -> weighted composite score -> sensitivity analysis

**Reads:** `london-final-light/outputs/phase3/` · **Writes:** `london-final-light/outputs/phase4/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr

print('Imports OK')

In [ ]:
# Loading Dataset
BASE = Path('Dataset path')
P3   = BASE / 'london-final-light' / 'outputs' / 'phase3'
OUT  = BASE / 'london-final-light' / 'outputs' / 'phase4'
OUT.mkdir(parents=True, exist_ok=True)

LOG_FEATURES = ['severity_weighted_count', 'ntl_mean_radiance']  # right-skewed -> log1p
print('Output folder:', OUT)

## Section-1: Load & Prepare Data
FEATURES are derived from the validated parquet. Any `*_rank` deprivation column is inverted so that a higher value = more deprived = higher demand.

In [ ]:
val = pd.read_parquet(P3 / 'phase3_validated_features.parquet')
print(f'Loaded: {val.shape}')
print(f'Columns: {list(val.columns)}')

META = ['lsoa21cd', 'lsoa21nm', 'lad22nm', 'crime_count']
raw_feats = [c for c in val.columns if c not in META]

# Invert rank columns (rank 1 = most deprived) into a deprivation score
FEATURES = []
for f in raw_feats:
    if f.endswith('_rank'):
        dep = f.replace('_rank', '_deprivation')
        mx = val[f].max()
        val[dep] = mx + 1 - val[f]
        FEATURES.append(dep)
        print(f'Inverted {f} -> {dep} (range {val[dep].min():.0f}–{val[dep].max():.0f})')
    else:
        FEATURES.append(f)

print(f'\nModel FEATURES: {FEATURES}')
print(f'Nulls in features: {val[FEATURES + ["crime_count"]].isnull().sum().to_dict()}')

## 4a. Weight Derivation
### Step 1: Negative Binomial Regression (primary weights)
Crime counts are overdispersed count data Negative Binomial is the statistically correct model. Standardised coefficients become empirically justified weights with p-values.

In [ ]:
# Standardise features for comparable coefficients; log-transform skewed ones
X_raw = val[FEATURES].copy()
for col in LOG_FEATURES:
    if col in X_raw.columns:
        X_raw[col] = np.log1p(X_raw[col])

X_std = (X_raw - X_raw.mean()) / X_raw.std()
X_std_const = sm.add_constant(X_std)
y = val['crime_count'].astype(int)

log_y = np.log1p(y)
ols_start = sm.OLS(log_y, X_std_const).fit()
start_params = np.append(ols_start.params.values, 0.5)

print('Fitting Negative Binomial Regression (lbfgs, OLS start params)...')
nb_model = sm.NegativeBinomial(y, X_std_const)
nb_result = nb_model.fit(start_params=start_params, method='lbfgs', maxiter=2000, disp=False)
print('Done.')
print(nb_result.summary())

In [ ]:
nb_coef = pd.DataFrame({
    'feature':  FEATURES,
    'nb_coef':  [nb_result.params[f] for f in FEATURES],
    'nb_pval':  [nb_result.pvalues[f] for f in FEATURES],
})
nb_coef['significant'] = nb_coef['nb_pval'] < 0.05
nb_coef['direction']   = nb_coef['nb_coef'].apply(lambda x: 'positive' if x >= 0 else 'negative')

print('--- Negative Binomial Coefficients ---')
print(nb_coef.to_string(index=False))

flagged = nb_coef[~nb_coef['significant']]['feature'].tolist()
if flagged:
    print(f'\nFlagged (p > 0.05): {flagged} — documented but retained (Phase 3 validated)')
else:
    print('\nAll features significant at p < 0.05')

In [ ]:
# Anomaly detection: LSOAs where observed >> expected
predicted = nb_result.predict(X_std_const)
val['nb_predicted'] = predicted
val['nb_ratio']     = val['crime_count'] / val['nb_predicted'].clip(lower=1)

anomalies = val[val['nb_ratio'] > 2.0][['lsoa21cd','lsoa21nm','lad22nm','crime_count','nb_predicted','nb_ratio']]
anomalies = anomalies.sort_values('nb_ratio', ascending=False)
print(f'LSOAs where observed > 2x predicted (anomalies): {len(anomalies)}')
print(anomalies.head(10).to_string(index=False))

### Step 2: Random Forest Feature Importances (cross-validation of weights)

In [ ]:
print('Fitting Random Forest...')
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, max_features='sqrt')
rf.fit(X_raw, y)

rf_importances = pd.DataFrame({
    'feature':       FEATURES,
    'rf_importance': rf.feature_importances_,
}).sort_values('rf_importance', ascending=False)

cv_scores = cross_val_score(rf, X_raw, y, cv=5, scoring='r2')
print(f'RF 5-fold CV R2: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print('\n--- RF Feature Importances ---')
print(rf_importances.to_string(index=False))

### Step 3: Compare NB vs RF & Derive Final Weights

In [ ]:
weights_df = nb_coef.merge(rf_importances, on='feature')
weights_df['nb_abs_coef'] = weights_df['nb_coef'].abs()
weights_df['nb_weight']   = weights_df['nb_abs_coef'] / weights_df['nb_abs_coef'].sum()
weights_df['rf_weight']   = weights_df['rf_importance'] / weights_df['rf_importance'].sum()
weights_df['final_weight'] = (weights_df['nb_weight'] + weights_df['rf_weight']) / 2
weights_df['final_weight'] = weights_df['final_weight'] / weights_df['final_weight'].sum()

rank_corr, rank_p = spearmanr(weights_df['nb_weight'], weights_df['rf_weight'])

print('--- Weight Comparison: NB vs RF ---')
print(weights_df[['feature','nb_weight','rf_weight','final_weight','nb_pval','significant']].to_string(index=False))
print(f'\nNB vs RF rank correlation: r={rank_corr:.4f} (p={rank_p:.4f})')
print(f'Agreement: {"STRONG" if rank_corr > 0.7 else "MODERATE" if rank_corr > 0.4 else "WEAK"}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colours = ['#1D9E75','#378ADD','#D85A30']
titles  = ['Negative Binomial weights', 'Random Forest weights', 'Final blended weights']
wcols   = ['nb_weight', 'rf_weight', 'final_weight']
for ax, col, title, colour in zip(axes, wcols, titles, colours):
    sorted_df = weights_df.sort_values(col)
    ax.barh(sorted_df['feature'], sorted_df[col], color=colour, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Weight', fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    for i, v in enumerate(sorted_df[col]):
        ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8)
plt.suptitle('Feature weights — london-final-light: NB vs RF vs Final blend', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / '4a_weight_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 4a_weight_comparison.png')

## 4b. Build the Risk Index
### Step 4: Normalise Features to [0, 1]

In [ ]:
scaler = MinMaxScaler()
X_norm = pd.DataFrame(scaler.fit_transform(X_raw), columns=FEATURES, index=val.index)
print('Normalised feature stats (all should be in [0,1]):')
print(X_norm.describe().round(3).to_string())

### Step 5: Compute Weighted Composite Risk Score

In [ ]:
final_weights = weights_df.set_index('feature')['final_weight']
val['risk_score'] = sum(X_norm[feat] * final_weights[feat] for feat in FEATURES)
val['risk_score_scaled'] = (
    (val['risk_score'] - val['risk_score'].min()) /
    (val['risk_score'].max() - val['risk_score'].min()) * 100
).round(2)

print('Risk score distribution:')
print(val[['risk_score', 'risk_score_scaled']].describe().round(4))
print('\nTop 10 highest-risk LSOAs:')
print(val.nlargest(10, 'risk_score_scaled')[['lsoa21cd','lsoa21nm','lad22nm','crime_count','risk_score_scaled']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(val['risk_score_scaled'], bins=60, color='#378ADD', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Risk score (0–100)', fontsize=11)
axes[0].set_ylabel('Number of LSOAs', fontsize=11)
axes[0].set_title('Distribution of risk scores — london-final-light', fontsize=12)
axes[0].spines[['top','right']].set_visible(False)
axes[1].scatter(val['risk_score_scaled'], val['crime_count'], alpha=0.3, s=8, color='#D85A30')
axes[1].set_xlabel('Risk score (0–100)', fontsize=11)
axes[1].set_ylabel('Actual crime count', fontsize=11)
axes[1].set_title('Risk score vs actual crime count (validation)', fontsize=12)
axes[1].spines[['top','right']].set_visible(False)
r, _ = spearmanr(val['risk_score_scaled'], val['crime_count'])
axes[1].annotate(f'Spearman r = {r:.3f}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=10)
plt.tight_layout()
plt.savefig(OUT / '4b_risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 4b_risk_score_distribution.png')

### Step 6: Sensitivity Analysis
Vary each weight by +-10% and check rank stability of top/bottom 10% LSOAs.

In [ ]:
def compute_risk(X_norm, weights):
    w = pd.Series(weights)
    w = w / w.sum()
    score = sum(X_norm[f] * w[f] for f in FEATURES)
    return (score - score.min()) / (score.max() - score.min()) * 100

baseline_weights = final_weights.to_dict()
top10_idx    = set(val.nlargest(int(len(val) * 0.1), 'risk_score_scaled').index)
bottom10_idx = set(val.nsmallest(int(len(val) * 0.1), 'risk_score_scaled').index)

sensitivity_rows = []
for feat in FEATURES:
    for delta_pct, label in [(-0.10, '-10%'), (+0.10, '+10%')]:
        w_perturbed = baseline_weights.copy()
        w_perturbed[feat] *= (1 + delta_pct)
        perturbed_scores = compute_risk(X_norm, w_perturbed)
        temp = val.copy()
        temp['perturbed_score'] = perturbed_scores.values
        new_top10    = set(temp.nlargest(int(len(val) * 0.1), 'perturbed_score').index)
        new_bottom10 = set(temp.nsmallest(int(len(val) * 0.1), 'perturbed_score').index)
        sensitivity_rows.append({
            'feature': feat, 'perturbation': label,
            'top10_stability_%':    round(len(top10_idx & new_top10) / len(top10_idx) * 100, 1),
            'bottom10_stability_%': round(len(bottom10_idx & new_bottom10) / len(bottom10_idx) * 100, 1),
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print('--- Sensitivity Analysis ---')
print(sensitivity_df.to_string(index=False))
avg_top    = sensitivity_df['top10_stability_%'].mean()
avg_bottom = sensitivity_df['bottom10_stability_%'].mean()
print(f'\nMean top-10% stability:    {avg_top:.1f}%')
print(f'Mean bottom-10% stability: {avg_bottom:.1f}%')

In [ ]:
import seaborn as sns
pivot_top = sensitivity_df.pivot(index='feature', columns='perturbation', values='top10_stability_%')
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_top, annot=True, fmt='.1f', cmap='RdYlGn', vmin=80, vmax=100,
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of top-10% LSOAs retained'})
ax.set_title('Sensitivity — top-10% rank stability (+-10% weight) — london-final-light', fontsize=11)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(OUT / '4b_sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 4b_sensitivity_heatmap.png')

## Save Outputs

In [ ]:
risk_output = val[['lsoa21cd','lsoa21nm','lad22nm','crime_count',
                   'risk_score','risk_score_scaled','nb_predicted','nb_ratio'] + FEATURES].copy()
risk_output.to_parquet(OUT / 'phase4_risk_scores.parquet', index=False)
print(f'Saved phase4_risk_scores.parquet — {risk_output.shape}')

sensitivity_df.to_parquet(OUT / 'phase4_sensitivity_results.parquet', index=False)
print(f'Saved phase4_sensitivity_results.parquet — {sensitivity_df.shape}')

In [ ]:
lines = [
    '# Phase 4 — Weight Justification (london-final-light)',
    '',
    'london-final-light is an ablation of the main London model that **excludes `stop_search_rate` and `seasonal_volatility`**.',
    '',
    '## Method',
    'Final weights are the average of two independently derived weight sets:',
    '1. **Negative Binomial Regression** (primary) — standardised coefficients from a NB regression of crime_count on validated features. NB is used because crime counts are overdispersed count data (Laufs et al., 2021).',
    '2. **Random Forest importances** (cross-validation) — feature importances from a 300-tree RF regressor trained on the same features.',
    f'   RF 5-fold CV R2: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}',
    f'   NB vs RF rank correlation: r={rank_corr:.4f}',
    '',
    '## Final Weights',
    '',
    '| Feature | NB weight | RF weight | Final weight | NB p-value | Significant |',
    '|---|---|---|---|---|---|',
]
for _, row in weights_df.sort_values('final_weight', ascending=False).iterrows():
    lines.append(f"| {row['feature']} | {row['nb_weight']:.4f} | {row['rf_weight']:.4f} | {row['final_weight']:.4f} | {row['nb_pval']:.4f} | {'Yes' if row['significant'] else 'No*'} |")
lines += [
    '',
    '*Features flagged as non-significant are retained because they passed Phase 3 Spearman validation and are theoretically justified by the literature.',
    '',
    '## Features Excluded from this Model',
    '- `stop_search_rate` — excluded on ethical grounds (potential for compounding racial bias in demand estimates).',
    '- `seasonal_volatility` — excluded by group decision (seasonality retained only as diagnostic context in Phase 3).',
    '',
    '## Feature Direction',
    '- All features normalised to [0,1] min-max before weighting.',
    '- Any `*_rank` deprivation column was INVERTED (deprivation = max_rank + 1 - rank) so higher value = more deprived = higher demand.',
    '- All other features are positively associated with policing demand.',
    '',
    '## Sensitivity Analysis',
    f'- Mean top-10% rank stability across all +-10% perturbations: **{avg_top:.1f}%**',
    f'- Mean bottom-10% rank stability: **{avg_bottom:.1f}%**',
    '- High stability confirms the index is robust to small changes in weights.',
    '',
    '## Literature Cross-Reference',
    '- Laufs et al. (2021) identify crime volume, severity, and deprivation as the primary drivers of police demand.',
    '- Cambridge Crime Harm Index (Sherman et al., 2016) underpins the severity_weighted_count feature.',
]
with open(OUT / 'phase4_weight_justification.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('Saved phase4_weight_justification.md')

In [ ]:
print('=' * 55)
print('PHASE 4 SUMMARY (london-final-light)')
print('=' * 55)
print(f'LSOAs scored:               {len(val)}')
print(f'Features in index:          {FEATURES}')
print(f'Risk score range:           {val["risk_score_scaled"].min():.1f} – {val["risk_score_scaled"].max():.1f}')
print(f'NB vs RF agreement (r):     {rank_corr:.4f}')
print(f'RF 5-fold CV R2:            {cv_scores.mean():.4f}')
print(f'Top-10% rank stability:     {avg_top:.1f}%')
print(f'Anomalous LSOAs (obs>2xexp):{len(anomalies)}')
print('=' * 55)
print(f'Outputs saved to: {OUT}')